In [4]:
from transformers import AutoTokenizer

# 1. Load the tokenizer from the hub (No GPU required!)
model_id = "unsloth/gemma-4-E2B-it" # Or your specific Gemma-4 variant
tokenizer = AutoTokenizer.from_pretrained(model_id)

[huggingface_hub.utils._http|WARNING]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
C:\Users\Ashutosh\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ashutosh\.cache\huggingface\hub\models--unsloth--gemma-4-E2B-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/window

In [6]:
import json
from datasets import load_dataset


DATASET_PATH = "medlens_train.jsonl"


raw_data = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(raw_data)} examples (Arrow-backed, not in RAM)")

Generating train split: 943197 examples [00:00, 1050635.90 examples/s]

Loaded 943197 examples (Arrow-backed, not in RAM)


In [7]:
# Look at a sample
print("Sample training example:")
print(json.dumps(raw_data[0], indent=2))

Sample training example:
{
  "messages": [
    {
      "role": "user",
      "content": "Review this medication regimen for interaction risk. Patient: a 54 year-old male. Medications: azathioprine, cyclosporine, mycophenolate mofetil, prednisone. Indications: Immunosuppressant drug therapy, Immunosuppression, Product used for unknown indication. Symptoms: Asthenia, Confusional state, Glioblastoma multiforme. Give the likely interaction severity and why it stands out."
    },
    {
      "role": "assistant",
      "content": "MAJOR interaction risk. This regimen matches a major multi-drug adverse-event signal in FAERS. Reported events: Asthenia, Confusional state, Glioblastoma multiforme. Regimen reviewed: azathioprine, cyclosporine, mycophenolate mofetil, prednisone. FAERS supports the regimen-level signal across 6 possible drug pairs, but it does not isolate one confirmed causal pair. Urgent medication review is warranted."
    }
  ]
}


In [8]:
from datasets import load_dataset

SPLIT_SEED = 3407
EVAL_RATIO = 0.10
MAX_SEQ_LENGTH = 2048
SAVE_PATH = "medlens_tokenized"

raw_dataset = raw_data

def normalize_messages(messages):
    normalized = []
    for message in messages:
        role = message.get("role")
        content = message.get("content", "")

        if isinstance(content, list):
            text_parts = []
            for item in content:
                if isinstance(item, dict):
                    if item.get("type") == "text":
                        text_parts.append(item.get("text", ""))
                elif isinstance(item, str):
                    text_parts.append(item)
            content = "\n".join(part for part in text_parts if part)
        elif not isinstance(content, str):
            content = str(content)

        normalized.append({"role": role, "content": content})
    return normalized

def tokenize_example(example):
    messages = example.get("messages")
    if not messages or not isinstance(messages, list):
        return {"input_ids": [], "attention_mask": []}

    normalized_messages = normalize_messages(messages)
    text = tokenizer.apply_chat_template(
        normalized_messages, tokenize=False, add_generation_prompt=False
    )

    if not isinstance(text, str):
        return {"input_ids": [], "attention_mask": []}

    text = text.removeprefix("<bos>").strip()
    if not text:
        return {"input_ids": [], "attention_mask": []}

    tok = tokenizer(text=text, truncation=True, max_length=MAX_SEQ_LENGTH)
    return {"input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"]}

sample_tokenized = tokenize_example(raw_dataset[0])
if not sample_tokenized["input_ids"]:
    raise RuntimeError("Sample tokenization produced no tokens. Check message schema.")

# Tokenize one example at a time (not batched) to avoid worker crashes
tokenized = raw_dataset.map(
    tokenize_example,
    remove_columns=raw_dataset.column_names,
    desc="Tokenizing",
)

# Drop empties (skipped examples)
tokenized = tokenized.filter(lambda x: len(x["input_ids"]) > 0)
if len(tokenized) == 0:
    raise RuntimeError(
        "Tokenization produced 0 rows. Verify the tokenizer/chat template cells ran successfully."
    )

# Split
dataset_split = tokenized.train_test_split(
    test_size=EVAL_RATIO, seed=SPLIT_SEED, shuffle=True
)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# Save to disk so you never redo this
train_dataset.save_to_disk(f"{SAVE_PATH}/train")
eval_dataset.save_to_disk(f"{SAVE_PATH}/eval")

print(f"Tokenized: {len(tokenized)} | Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"Saved to {SAVE_PATH}/ — next time just load from disk")

Saving the dataset (1/1 shards): 100%|██████████| 94320/94320 [00:00<00:00, 214123.20 examples/s]

Tokenized: 943197 | Train: 848877 | Eval: 94320
Saved to medlens_tokenized/ — next time just load from disk


In [9]:
# Load train and eval datasets from disk
from datasets import load_from_disk

train_dataset = load_from_disk(f"{SAVE_PATH}/train")
eval_dataset = load_from_disk(f"{SAVE_PATH}/eval")

if len(train_dataset) == 0 or len(eval_dataset) == 0:
    raise RuntimeError(
        "Saved datasets are empty. Re-run the tokenization cell after loading the tokenizer/chat template."
    )

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

Train: 848877 | Eval: 94320


In [12]:
# train_dataset[0]